# Assignment 2: Advanced RAG Techniques
## Day 6 Session 2 - Advanced RAG Fundamentals

**OBJECTIVE:** Implement advanced RAG techniques including postprocessors, response synthesizers, and structured outputs.

**LEARNING GOALS:**
- Understand and implement node postprocessors for filtering and reranking
- Learn different response synthesis strategies (TreeSummarize, Refine)
- Create structured outputs using Pydantic models
- Build advanced retrieval pipelines with multiple processing stages

**DATASET:** Use the same data folder as Assignment 1 (`Day_6/session_2/data/`)

**PREREQUISITES:** Complete Assignment 1 first

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it
3. The answers can be found in the `03_advanced_rag_techniques.ipynb` notebook
4. Each technique builds on the previous one


In [2]:
!pip install llama-index llama-index-llms-openai llama-index-embeddings-openai llama-index-embeddings-huggingface llama-index-vector-stores-lancedb lancedb pypdf llama-index-readers-file -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 11.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip install llama-index llama-index-llms-openrouter llama-index-embeddings-huggingface llama-index-vector-stores-lancedb lancedb pypdf llama-index-readers-file -q

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_rounds=max_rounds)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [1]:
!pip install llama-index llama-index-llms-openrouter llama-index-embeddings-openai llama-index-vector-stores-lancedb lancedb -q --no-warn-script-location

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 16.6 MB/s eta 0:00:00


In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path
from typing import List, Optional
from pydantic import BaseModel, Field

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers import TreeSummarize, CompactAndRefine

print("✅ Advanced RAG libraries imported successfully!")

✅ Advanced RAG libraries imported successfully!


In [18]:
def setup_advanced_rag_settings():
    api_key = os.getenv("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")
    if not api_key:
        print("No API key found")
    else:
        print("API key found")
        Settings.llm = OpenRouter(
            api_key=api_key,
            model="openai/gpt-4o-mini",
            temperature=0.1
        )
    Settings.embed_model = OpenAIEmbedding(
        model="text-embedding-3-small",
        api_key=api_key,
        api_base="https://openrouter.ai/api/v1"
    )
    Settings.chunk_size = 512
    Settings.chunk_overlap = 50
    print("Settings configured successfully")

setup_advanced_rag_settings()


No API key found
Settings configured successfully


In [29]:
from google.colab import userdata
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [30]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
import os

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1"
)
print("Embeddings configured!")

Embeddings configured!


In [35]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.schema import Document
from llama_index.vector_stores.lancedb import LanceDBVectorStore

raw_documents = SimpleDirectoryReader(
    "/content/ai-accelerator-C5/Day_6/session_2/data/",
    recursive=True,
    exclude=["*.mp3", "*.mp4", "*.wav", "*.m4a"],
    errors="ignore"
).load_data()

# Create new clean documents
documents = [
    Document(
        text=doc.text.encode("ascii", errors="ignore").decode("ascii"),
        metadata=doc.metadata
    )
    for doc in raw_documents
]

print(f"Loaded and cleaned {len(documents)} documents")

vector_store = LanceDBVectorStore(
    uri="./advanced_rag_vectordb",
    table_name="documents",
    mode="overwrite"
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)
print(f"Index built successfully!")

Loaded and cleaned 39 documents


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/97 [00:00<?, ?it/s]

UnicodeEncodeError: 'ascii' codec can't encode character '\xa0' in position 7: ordinal not in range(128)

In [34]:
doc.text = doc.text.encode("ascii", errors="ignore").decode("ascii")

AttributeError: property 'text' of 'Document' object has no setter

In [21]:
# Setup: Create index from Assignment 1 (reuse the basic functionality)
def setup_basic_index(data_folder: str = "/content/ai-accelerator-C5/Day_6/session_2/data/", force_rebuild: bool = False):
    """
    Create a basic vector index that we'll enhance with advanced techniques.
    This reuses the concepts from Assignment 1.
    """
    # Create vector store
    vector_store = LanceDBVectorStore(
        uri="./advanced_rag_vectordb",
        table_name="documents"
    )

    # Load documents
    if not Path(data_folder).exists():
        print(f"❌ Data folder not found: {data_folder}")
        return None

    reader = SimpleDirectoryReader(input_dir=data_folder, recursive=True)
    documents = reader.load_data()

    # Create storage context and index
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )

    print(f"✅ Basic index created with {len(documents)} documents")
    print("   Ready for advanced RAG techniques!")
    return index

# Create the basic index
print("📁 Setting up basic index for advanced RAG...")
index = setup_basic_index()

if index:
    print("🚀 Ready to implement advanced RAG techniques!")
else:
    print("❌ Failed to create index - check data folder path")


📁 Setting up basic index for advanced RAG...
❌ Data folder not found: /content/ai-accelerator-C5/Day_6/session_2/data/
❌ Failed to create index - check data folder path


In [49]:
import os
from google.colab import userdata

clean_key = userdata.get("OPENROUTER_API_KEY").strip()
os.environ["OPENROUTER_API_KEY"] = clean_key
print(f"Key length: {len(clean_key)}")
print(f"Last 5 chars repr: {repr(clean_key[-5:])}")

Key length: 73
Last 5 chars repr: 'ea180'


In [51]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=clean_key,
    api_base="https://openrouter.ai/api/v1"
)
print("Done!")

Done!


In [39]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=clean_key,
    api_base="https://openrouter.ai/api/v1"
)
print("Embeddings reconfigured with clean key!")

Embeddings reconfigured with clean key!


In [57]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.schema import Document
from llama_index.vector_stores.lancedb import LanceDBVectorStore

raw_documents = SimpleDirectoryReader(
    "/content/ai-accelerator-C5/Day_6/session_2/data/",
    recursive=True,
    exclude=["*.mp3", "*.mp4", "*.wav", "*.m4a"],
    errors="ignore"
).load_data()

documents = [
    Document(
        text=doc.text.encode("ascii", errors="ignore").decode("ascii"),
        metadata=doc.metadata
    )
    for doc in raw_documents
]

print(f"Loaded and cleaned {len(documents)} documents")

vector_store = LanceDBVectorStore(
    uri="./advanced_rag_vectordb",
    table_name="documents",
    mode="overwrite"
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)
print("Index built successfully!")

Loaded and cleaned 39 documents


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/97 [00:00<?, ?it/s]

Index built successfully!


In [60]:
from google.colab import userdata
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY").strip()
print(len(os.environ["OPENROUTER_API_KEY"]))

SecretNotFoundError: Secret OPENROUTER_API_KEY does not exist.

In [45]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
import os

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1"
)
print("Embeddings ready!")

Embeddings ready!


## 1. Node Postprocessors - Similarity Filtering

**Concept:** Postprocessors refine retrieval results after the initial vector search. The `SimilarityPostprocessor` filters out chunks that fall below a relevance threshold.

**Why it matters:** Raw vector search often returns some irrelevant results. Filtering improves precision and response quality.

Complete the function below to create a query engine with similarity filtering.


In [47]:
def create_query_engine_with_similarity_filter(index, similarity_cutoff=0.3, top_k=10):
    similarity_processor = SimilarityPostprocessor(similarity_cutoff=similarity_cutoff)
    query_engine = index.as_query_engine(
        similarity_top_k=top_k,
        node_postprocessors=[similarity_processor]
    )
    print(f"Query engine created with cutoff={similarity_cutoff}, top_k={top_k}")
    return query_engine

if index:
    filtered_engine = create_query_engine_with_similarity_filter(index, similarity_cutoff=0.3)
    if filtered_engine:
        print("✅ Query engine with similarity filtering created")
        test_query = "What are the benefits of AI agents?"
        print(f"Testing query: '{test_query}'")
        response = filtered_engine.query(test_query)
        print(f"Response: {response}")
    else:
        print("❌ Failed to create filtered query engine")
else:
    print("❌ No index available - run previous cells first")

❌ No index available - run previous cells first


In [52]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.schema import Document
from llama_index.vector_stores.lancedb import LanceDBVectorStore

raw_documents = SimpleDirectoryReader(
    "/content/ai-accelerator-C5/Day_6/session_2/data/",
    recursive=True,
    exclude=["*.mp3", "*.mp4", "*.wav", "*.m4a"],
    errors="ignore"
).load_data()

documents = [
    Document(
        text=doc.text.encode("ascii", errors="ignore").decode("ascii"),
        metadata=doc.metadata
    )
    for doc in raw_documents
]

print(f"Loaded and cleaned {len(documents)} documents")

vector_store = LanceDBVectorStore(
    uri="./advanced_rag_vectordb",
    table_name="documents",
    mode="overwrite"
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)
print("Index built successfully!")

Loaded and cleaned 39 documents


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/97 [00:00<?, ?it/s]

Index built successfully!


In [61]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.schema import Document
from llama_index.vector_stores.lancedb import LanceDBVectorStore

raw_documents = SimpleDirectoryReader(
    "/content/ai-accelerator-C5/Day_6/session_2/data/",
    recursive=True,
    exclude=["*.mp3", "*.mp4", "*.wav", "*.m4a"],
    errors="ignore"
).load_data()

documents = [
    Document(
        text=doc.text.encode("ascii", errors="ignore").decode("ascii"),
        metadata=doc.metadata
    )
    for doc in raw_documents
]

print(f"Loaded and cleaned {len(documents)} documents")

vector_store = LanceDBVectorStore(
    uri="./advanced_rag_vectordb",
    table_name="documents",
    mode="overwrite"
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)
print("Index built successfully!")

Loaded and cleaned 39 documents


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/97 [00:00<?, ?it/s]

Index built successfully!


## 2. Response Synthesizers - TreeSummarize

**Concept:** Response synthesizers control how retrieved information becomes final answers. `TreeSummarize` builds responses hierarchically, ideal for complex analytical questions.

**Why it matters:** Different synthesis strategies work better for different query types. TreeSummarize excels at comprehensive analysis and long-form responses.

Complete the function below to create a query engine with TreeSummarize response synthesis.


In [62]:
def create_query_engine_with_tree_summarize(index, top_k: int = 5):
    """
    Create a query engine that uses TreeSummarize for comprehensive responses.

    TODO: Complete this function to create a query engine with TreeSummarize synthesis.
    HINT: Create a TreeSummarize instance, then use index.as_query_engine() with response_synthesizer parameter

    Args:
        index: Vector index to query
        top_k: Number of results to retrieve

    Returns:
        Query engine with TreeSummarize synthesis
    """
    # TODO: Create TreeSummarize response synthesizer
    # tree_synthesizer =

    # TODO: Create query engine with the synthesizer
    # query_engine =

    # return query_engine

    # PLACEHOLDER - Replace with actual implementation
    print(f"TODO: Create query engine with TreeSummarize synthesis")
    return None

# Test the function
if index:
    tree_engine = create_query_engine_with_tree_summarize(index)

    if tree_engine:
        print("✅ Query engine with TreeSummarize created")

        # Test with a complex analytical query
        analytical_query = "Compare the advantages and disadvantages of different AI agent frameworks"
        print(f"\n🔍 Testing analytical query: '{analytical_query}'")

        # Uncomment when implemented:
        # response = tree_engine.query(analytical_query)
        # print(f"📝 TreeSummarize Response:\n{response}")
        print("   (Complete the function above to test comprehensive analysis)")
    else:
        print("❌ Failed to create TreeSummarize query engine")
else:
    print("❌ No index available - run previous cells first")


TODO: Create query engine with TreeSummarize synthesis
❌ Failed to create TreeSummarize query engine


## 3. Structured Outputs with Pydantic Models

**Concept:** Structured outputs ensure predictable, parseable responses using Pydantic models. This is essential for API endpoints and data pipelines.

**Why it matters:** Instead of free-text responses, you get type-safe, validated data structures that applications can reliably process.

Complete the function below to create a structured output system for extracting research paper information.


In [63]:
# First, define the Pydantic models for structured outputs
class ResearchPaperInfo(BaseModel):
    """Structured information about a research paper or AI concept."""
    title: str = Field(description="The main title or concept name")
    key_points: List[str] = Field(description="3-5 main points or findings")
    applications: List[str] = Field(description="Practical applications or use cases")
    summary: str = Field(description="Brief 2-3 sentence summary")

# Import the missing component
from llama_index.core.program import LLMTextCompletionProgram

def create_structured_output_program(output_model: BaseModel = ResearchPaperInfo):
    """
    Create a structured output program using Pydantic models.

    TODO: Complete this function to create a structured output program.
    HINT: Use LLMTextCompletionProgram.from_defaults() with PydanticOutputParser and a prompt template

    Args:
        output_model: Pydantic model class for structured output

    Returns:
        LLMTextCompletionProgram that returns structured data
    """
    # TODO: Create output parser with the Pydantic model
    # output_parser = ?

    # TODO: Create the structured output program
    # program = ?

    # return program

    # PLACEHOLDER - Replace with actual implementation
    print(f"TODO: Create structured output program with {output_model.__name__}")
    return None

# Test the function
if index:
    structured_program = create_structured_output_program(ResearchPaperInfo)

    if structured_program:
        print("✅ Structured output program created")

        # Test with retrieval and structured extraction
        structure_query = "Tell me about AI agents and their capabilities"
        print(f"\n🔍 Testing structured query: '{structure_query}'")

        # Get context for structured extraction (Uncomment when implemented)
        # retriever = VectorIndexRetriever(index=index, similarity_top_k=3)
        # nodes = retriever.retrieve(structure_query)
        # context = "\n".join([node.text for node in nodes])

        # Uncomment when implemented:
        # response = structured_program(context=context, query=structure_query)
        # print(f"📊 Structured Response:\n{response}")
        print("   (Complete the function above to get structured JSON output)")

        print("\n💡 Expected output format:")
        print("   - title: String")
        print("   - key_points: List of strings")
        print("   - applications: List of strings")
        print("   - summary: String")
    else:
        print("❌ Failed to create structured output program")
else:
    print("❌ No index available - run previous cells first")


TODO: Create structured output program with ResearchPaperInfo
❌ Failed to create structured output program


## 4. Advanced Pipeline - Combining All Techniques

**Concept:** Combine multiple advanced techniques into a single powerful query engine: similarity filtering + response synthesis + structured output.

**Why it matters:** Production RAG systems often need multiple techniques working together for optimal results.

Complete the function below to create a comprehensive advanced RAG pipeline.


In [64]:
def create_advanced_rag_pipeline(index, similarity_cutoff: float = 0.3, top_k: int = 10):
    """
    Create a comprehensive advanced RAG pipeline combining multiple techniques.

    TODO: Complete this function to create the ultimate advanced RAG query engine.
    HINT: Combine SimilarityPostprocessor + TreeSummarize using index.as_query_engine()

    Args:
        index: Vector index to query
        similarity_cutoff: Minimum similarity score for filtering
        top_k: Number of initial results to retrieve

    Returns:
        Advanced query engine with filtering and synthesis combined
    """
    # TODO: Create similarity postprocessor
    # similarity_processor = ?

    # TODO: Create TreeSummarize for comprehensive responses
    # tree_synthesizer = ?

    # TODO: Create the comprehensive query engine combining both techniques
    # advanced_engine = ?

    # return advanced_engine

    # PLACEHOLDER - Replace with actual implementation
    print(f"TODO: Create advanced RAG pipeline with all techniques")
    return None

# Test the comprehensive pipeline
if index:
    advanced_pipeline = create_advanced_rag_pipeline(index)

    if advanced_pipeline:
        print("✅ Advanced RAG pipeline created successfully!")
        print("   🔧 Similarity filtering: ✅")
        print("   🌳 TreeSummarize synthesis: ✅")

        # Test with complex query
        complex_query = "Analyze the current state and future potential of AI agent technologies"
        print(f"\n🔍 Testing complex query: '{complex_query}'")

        # Uncomment when implemented:
        # response = advanced_pipeline.query(complex_query)
        # print(f"🚀 Advanced RAG Response:\n{response}")
        print("   (Complete the function above to test the full pipeline)")

        print("\n🎯 This should provide:")
        print("   - Filtered relevant results only")
        print("   - Comprehensive analytical response")
        print("   - Combined postprocessing and synthesis")
    else:
        print("❌ Failed to create advanced RAG pipeline")
else:
    print("❌ No index available - run previous cells first")


TODO: Create advanced RAG pipeline with all techniques
❌ Failed to create advanced RAG pipeline


## 5. Final Test - Compare Basic vs Advanced RAG

Once you've completed all the functions above, run this cell to compare basic RAG with your advanced techniques.


In [66]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.schema import Document
from llama_index.vector_stores.lancedb import LanceDBVectorStore

raw_documents = SimpleDirectoryReader(
    "/content/ai-accelerator-C5/Day_6/session_2/data/",
    recursive=True,
    exclude=["*.mp3", "*.mp4", "*.wav", "*.m4a"],
    errors="ignore"
).load_data()

documents = [
    Document(
        text=doc.text.encode("ascii", errors="ignore").decode("ascii"),
        metadata=doc.metadata
    )
    for doc in raw_documents
]

vector_store = LanceDBVectorStore(uri="./advanced_rag_vectordb", table_name="documents", mode="overwrite")
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context, show_progress=True)
print("Index built successfully!")


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/97 [00:00<?, ?it/s]

Index built successfully!
